# Persian Offensive Comment Detection

TF-IDF + LinearSVC binary classifier (`Offensive` vs `Neutral`) trained on `ParsOffensive.xlsx`.

## Load data

In [2]:
import pandas as pd

df=pd.read_excel("../data/ParsOffensive.xlsx")
df.head()

,comment,label
0,الان این فیلتر چه بادیگاردی هم براش گذاشتن یک ...,Offensive
1,سلام علیک سید جان,Neutral
2,سگ پدر,Offensive
3,خار چشم دشمنان ایران اسلامی سید عزیز,Neutral
4,سلام علیک بر سید عزیز,Neutral


In [3]:
import hazm

normalizer=hazm.Normalizer()
word_tokenizer=hazm.WordTokenizer()
lemmatizer=hazm.Lemmatizer()

def clean_stopword(stopword):
	return stopword.replace("آ", "ا")

stopwords=hazm.utils.stopwords_list()
stopwords=list(map(clean_stopword, stopwords))


for word in ["عالی", "خوب", "خوبی", "بی"]: stopwords.remove(word)

## Preprocessing setup

`hazm` normalizer/tokenizer/lemmatizer and a Persian stopword list, with sentiment-bearing words (`عالی`, `خوب`, `خوبی`, `بی`) kept out of the stopword list since they carry signal for this task.

In [4]:
import string
import emoji
import numpy as np

def remove_chars(text, chars):
	for char in chars: text=text.replace(char, " ")
	return text

def preprocess(text):
	if not isinstance(text, str): return np.nan

	text=text.lower()
	text=remove_chars(text, string.punctuation+"؟،")

	text=emoji.replace_emoji(text, replace=" ")

	normalized_txt=normalizer.normalize(text).replace("آ", "ا")
	normalized_txt=remove_chars(normalized_txt, "۰۱۲۳۴۵۶۷۸۹\u200c")
	
	tokens=word_tokenizer.tokenize(normalized_txt)

	tokens=list(map(
		lambda token: lemmatizer.lemmatize(token).split("#")[0],
		tokens
	))

	tokens=list(filter(
		lambda token: token not in stopwords,
		tokens
	))

	if len(tokens)==0: return np.nan

	return " ".join(tokens)


df.comment=df.comment.apply(preprocess)
df.head()

,comment,label
0,الان فیلتر بادیگاردی براش گذاشت وقت لال نشه,Offensive
1,سلام علیک سید جان,Neutral
2,سگ پدر,Offensive
3,خار چشم دشمن ایران اسلامی سید عزیز,Neutral
4,سلام علیک سید عزیز,Neutral


## Clean comments

Lowercase, strip punctuation/emoji/digits, normalize, tokenize, lemmatize, drop stopwords. Empty results become `NaN`.

In [5]:
print(df.shape)
df.drop_duplicates(subset="comment", keep="first", inplace=True)
df.shape

(8433, 2)


(7763, 2)

## Drop duplicates and nulls

In [6]:
print(df.isnull().sum())
df.dropna(inplace=True)
print(df.isnull().sum())
df.label.value_counts(normalize=True)

comment    1
label      0
dtype: int64
comment    0
label      0
dtype: int64


label
Neutral      0.642103
Offensive    0.357897
Name: proportion, dtype: float64

## Encode labels

In [7]:
import warnings
warnings.filterwarnings("ignore")

LABELS_DICT = {"Neutral": 0, "Offensive": 1}
df.label = df.label.map(LABELS_DICT).astype(int)
df.head()

,comment,label
0,الان فیلتر بادیگاردی براش گذاشت وقت لال نشه,1
1,سلام علیک سید جان,0
2,سگ پدر,1
3,خار چشم دشمن ایران اسلامی سید عزیز,0
4,سلام علیک سید عزیز,0


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
	df.comment, df.label,
	test_size=0.2,
	random_state=18
)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

pipeline = Pipeline([
	("vectorizer", TfidfVectorizer(ngram_range=(1, 2))),
	(
        "model",
        CalibratedClassifierCV(estimator=LinearSVC(class_weight="balanced"), method="sigmoid", cv=3)
    )
])

In [10]:
from sklearn.model_selection import GridSearchCV

param_grid = {
	"vectorizer__max_df": [0.75, 0.90, 1.0],
	"vectorizer__min_df": [1, 2, 5],
	"vectorizer__sublinear_tf": [True, False],
	"model__estimator__C": [0.1, 1, 10],
	"model__estimator__penalty": ["l2", "l1"],
	"model__estimator__loss": ["squared_hinge", "hinge"],
	"model__estimator__dual": [True, False],
}

grid = GridSearchCV(
	pipeline,
	param_grid,
	cv=5,
	scoring="accuracy",
	n_jobs=-1
)

grid.fit(X_train, y_train)
print(grid.best_score_)
print(grid.best_params_)

0.8452231266406371
{'model__estimator__C': 1, 'model__estimator__dual': False, 'model__estimator__loss': 'squared_hinge', 'model__estimator__penalty': 'l1', 'vectorizer__max_df': 0.9, 'vectorizer__min_df': 2, 'vectorizer__sublinear_tf': False}


In [14]:
pipeline = Pipeline([
	("vectorizer", TfidfVectorizer(
		ngram_range=(1, 2),
        max_df=0.9,
        min_df=2,
        sublinear_tf=True
	)),
	(
        "model",
        CalibratedClassifierCV(
            estimator=LinearSVC(
				class_weight="balanced",
                C=1,
                dual=False,
                loss="squared_hinge",
                penalty="l1"
            ),
            method="sigmoid",
            cv=3
		)
	)
])

pipeline.fit(X_train, y_train)
pipeline.score(X_test, y_test)

0.8634900193174501

In [12]:
pipeline.fit(df.comment, df.label)

import joblib

joblib.dump(
    pipeline,
    "../web/services/models/offensive_detection_pipeline.joblib"
)

['../web/services/models/offensive_detection_pipeline.joblib']